In [1]:
%%bash
apt-get -y install libopenjp2-7-dev libopenjp2-tools openslide-tools libpixman-1-dev | tail -n 1
pip install git+https://github.com/TissueImageAnalytics/tiatoolbox.git@develop | tail -n 1
pip install timm huggingface_hub torchvision
apt update && apt install -y openslide-tools
pip install openslide-python
echo "Installation is done."


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,306 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,651 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-se

  Running command git clone --filter=blob:none --quiet https://github.com/TissueImageAnalytics/tiatoolbox.git /tmp/pip-req-build-3e8jypc_
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==5.5.6, but you have ipykernel 6.29.5 which is incompatible.
notebook 6.5.5 requires jupyter-client<8,>=5.3.4, but you have jupyter-client 8.6.3 which is incompatible.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)




In [2]:
"""Import modules required to run the Jupyter notebook."""

# Clear logger to use tiatoolbox.logger
import logging

if logging.getLogger().hasHandlers():
    logging.getLogger().handlers.clear()

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import requests

from tiatoolbox import logger
from tiatoolbox.tools import patchextraction
from tiatoolbox.utils.misc import imread, read_locations
from tiatoolbox.wsicore.wsireader import WSIReader

mpl.rcParams["figure.dpi"] = 300  # for high resolution figure in notebook
mpl.rcParams["figure.facecolor"] = "white"  # To make sure text is visible in dark mode

|2025-02-01|15:10:34.269| [INFO] NumExpr defaulting to 12 threads.


|2025-02-01|15:11:04.578| [WARNING] /usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.2 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()



In [3]:
![ -d tmp ] && ( echo "deleting tmp directory"; rm -rf tmp )

In [4]:
from tiatoolbox.models.engine.semantic_segmentor import (
    DeepFeatureExtractor,
    IOSegmentorConfig,
)
from tiatoolbox.models.architecture.vanilla import TimmBackbone
import os

os.makedirs("feature_extraction", exist_ok=True)
global_save_dir = Path("feature_extraction")
device = "cuda"

wsi_path = "/content/drive/MyDrive/Perso/SLA/Asclepios/svs_slides/image_test.svs"

model = TimmBackbone(backbone="H-optimus-0", pretrained=True)

wsi_ioconfig = IOSegmentorConfig(
    input_resolutions=[{"units": "mpp", "resolution": 0.5}],
    patch_input_shape=[224, 224],
    output_resolutions=[{"units": "mpp", "resolution": 0.5}],
    patch_output_shape=[224, 224],
    stride_shape=[224, 224],
)

# create the feature extractor and run it on the WSI
extractor = DeepFeatureExtractor(
    model=model,
    auto_generate_mask=True,
    batch_size=32,
    num_loader_workers=4,
    num_postproc_workers=4,
)

out = extractor.predict(
    imgs=[wsi_path],
    mode="wsi",
    ioconfig=wsi_ioconfig,
    save_dir=global_save_dir / "wsi_features",
    device=device,
)

config.json:   0%|          | 0.00/447 [00:00<?, ?B/s]

|2025-02-01|15:11:27.281| [INFO] Loading pretrained weights from Hugging Face hub (bioptimus/H-optimus-0)


pytorch_model.bin:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

|2025-02-01|15:13:19.494| [WARNING] GPU is not compatible with torch.compile. Compatible GPUs include NVIDIA V100, A100, and H100. Speedup numbers may be lower than expected.
|2025-02-01|15:13:27.594| [WARNING] Read: Scale > 1.This means that the desired resolution is higher than the WSI baseline (maximum encoded resolution). Interpolation of read regions may occur.
Process Batch: 100%|##########################| 193/193 [08:27<00:00,  2.63s/it]

|2025-02-01|15:21:56.446| [INFO] Finish: 0
|2025-02-01|15:21:56.447| [INFO] --Input: /content/drive/MyDrive/Perso/SLA/Asclepios/svs_slides/image_test.svs
|2025-02-01|15:21:56.448| [INFO] --Output: /content/feature_extraction/wsi_features/0


In [5]:
!cp -r /content/feature_extraction/wsi_features /content/drive/MyDrive/Perso/SLA/Asclepios


In [ ]:
# First we define a function to calculate the umap reduction
import numpy as np
import umap

def umap_reducer(x: np.ndarray, dims: int = 3, nns: int = 10) -> np.ndarray:
    """UMAP reduction of the input data."""
    reducer = umap.UMAP(
        n_neighbors=nns,
        n_components=dims,
        metric="manhattan",
        spread=0.5,
        random_state=2,
    )
    reduced = reducer.fit_transform(x)
    reduced -= reduced.min(axis=0)
    reduced /= reduced.max(axis=0)
    return reduced


# load the features output by our feature extractor
pos = np.load(global_save_dir / "wsi_features" / "0.position.npy")
feats = np.load(global_save_dir / "wsi_features" / "0.features.0.npy")
pos = pos / 8  # as we extracted at 0.5mpp, and we are overlaying on a thumbnail at 4mpp

# reduce the features into 3 dimensional (rgb) space
reduced = umap_reducer(feats)

overview_resolution = (
    4  # the resolution in which we desire to merge and visualize the patch predictions
)
# the unit of the `resolution` parameter. Can be "power", "level", "mpp", or "baseline"
overview_unit = "mpp"
wsi = WSIReader.open(wsi_path)
wsi_overview = wsi.slide_thumbnail(resolution=overview_resolution, units=overview_unit)
plt.figure(), plt.imshow(wsi_overview)
plt.axis("off")

# plot the feature map reduction
plt.figure()
plt.imshow(wsi_overview)
plt.scatter(pos[:, 0], pos[:, 1], c=reduced, s=1, alpha=0.5)
plt.axis("off")
plt.title("UMAP feature embedding")